# VAE Loss Function Experiments
**Tests:** LDS, FDS, Focal, BMSE | **Conclusion:** DIR losses don't help VAE

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')

# Path setup
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Models and utilities from project structure
from models.vae import SideslipVAE
from utils.data_loadingV2 import load_data
from utils.band_mapping import load_band_mapping, calculate_band_mae
from utils.simple_lds_loss import create_lds_weights
from utils.bmse_loss import BMCLoss
from utils.reproducibility_utils import seed_everything
from utils.visualization import plot_sweep_results, plot_method_comparison

SEED = 42
seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Load data
TRAIN_PATH = PROJECT_ROOT / "datasets_challenge" / "train"
COLUMNS_TO_DROP = ['sideSlip', 'vyCG', 'longitude', 'latitude', 'time']

train_loader, val_loader, scaler, _, _ = load_data(
    train_path=str(TRAIN_PATH), val_ratio=0.15, columns_to_drop=COLUMNS_TO_DROP,
    sequence_length=20, target="sideSlip", threshold=True, threshold_value=3.0,
    threshold_column="vxCG", batchsize=64, scaling=True, seed=SEED, shuffle=True
)
band_dict = load_band_mapping()

# Extract training targets for LDS weights
train_targets = []
for X, y in train_loader:
    train_targets.extend(y.numpy())
targets_flat = np.array(train_targets)
print(f"Train: {len(train_loader)} batches, Val: {len(val_loader)} batches")

In [ ]:
class VAEDIRLoss(nn.Module):
    """VAE loss with DIR components."""
    def __init__(self, beta_kl=0.005, dir_type='l1', weights=None):
        super().__init__()
        self.beta_kl, self.dir_type, self.lds_weights = beta_kl, dir_type, weights
        if dir_type == 'bmc': self.bmc_loss = BMCLoss(init_noise_sigma=1.0)
    
    def forward(self, outputs, x_input, y_true):
        y_pred, mu, logvar = outputs['y_pred'], outputs['mu'], outputs['logvar']
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        
        if self.dir_type == 'l1':
            pred_loss = F.l1_loss(y_pred, y_true)
        elif self.dir_type == 'weighted_l1':
            pred_loss = (torch.abs(y_pred - y_true) * (1.0 + torch.abs(y_true))).mean()
        elif self.dir_type == 'lds' and self.lds_weights is not None:
            indices = ((y_true + 5) / 10 * 100).long().clamp(0, len(self.lds_weights)-1)
            weights = torch.tensor([self.lds_weights[i] for i in indices]).to(device)
            pred_loss = (torch.abs(y_pred - y_true) * weights).mean()
        elif self.dir_type == 'focal':
            abs_diff = torch.abs(y_pred - y_true)
            pred_loss = (abs_diff * torch.pow(abs_diff + 0.1, 2.0)).mean()
        elif self.dir_type == 'bmc':
            pred_loss = self.bmc_loss(y_pred, y_true)
        else:
            pred_loss = F.l1_loss(y_pred, y_true)
        
        return pred_loss + self.beta_kl * kl_loss, {'pred': pred_loss.item(), 'kl': kl_loss.item()}

In [ ]:
def train_vae_with_loss(loss_type, train_loader, val_loader, band_dict, lds_weights=None):
    """Train VAE with specific loss."""
    model = SideslipVAE(input_dim=22, hidden_dim=64, latent_dim=16, seq_len=20).to(device)
    loss_fn = VAEDIRLoss(beta_kl=0.005, dir_type=loss_type, weights=lds_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    
    best_mae, best_state = float('inf'), None
    for epoch in range(50):
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss, _ = loss_fn(model(X), X, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        
        if (epoch + 1) % 10 == 0:
            metrics = evaluate(model, val_loader, band_dict)
            if metrics['overall_mae'] < best_mae:
                best_mae = metrics['overall_mae']
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    if best_state: model.load_state_dict(best_state)
    return model, evaluate(model, val_loader, band_dict)

@torch.no_grad()
def evaluate(model, loader, band_dict):
    model.eval()
    preds, targets = [], []
    for X, y in loader:
        preds.extend(model(X.to(device))['y_pred'].cpu().numpy().flatten())
        targets.extend(y.numpy())
    return calculate_band_mae(np.array(preds), np.array(targets), band_dict)

## 1. Loss Function Comparison

In [ ]:
lds_weights = create_lds_weights(targets_flat, kernel='gaussian', ks=5, sigma=2)

loss_results = []
for loss_type in ['l1', 'weighted_l1', 'lds', 'focal', 'bmc']:
    print(f"Training {loss_type}...")
    weights = lds_weights if loss_type == 'lds' else None
    _, metrics = train_vae_with_loss(loss_type, train_loader, val_loader, band_dict, weights)
    print(f"  Overall={metrics['overall_mae']:.4f}, Few={metrics['few_mae']:.4f}")
    loss_results.append({'loss': loss_type, **metrics})

pd.DataFrame(loss_results)

## 2. LDS Kernel Sweep

In [ ]:
lds_results = []
kernels = [('gaussian', 3, 1), ('gaussian', 5, 2), ('laplacian', 5, 2), ('triang', 5, None)]

for kernel, ks, sigma in kernels:
    config_name = f"{kernel}_ks{ks}_s{sigma}"
    print(f"Training {config_name}...")
    weights = create_lds_weights(targets_flat, kernel=kernel, ks=ks, sigma=sigma)
    _, metrics = train_vae_with_loss('lds', train_loader, val_loader, band_dict, weights)
    print(f"  Overall={metrics['overall_mae']:.4f}")
    lds_results.append({'config': config_name, **metrics})

pd.DataFrame(lds_results)

## 3. Ensemble Experiment

In [ ]:
models, all_preds = [], []
for i, loss_type in enumerate(['l1', 'weighted_l1', 'focal']):
    seed_everything(SEED + i)
    print(f"Training ensemble member: {loss_type}...")
    model, _ = train_vae_with_loss(loss_type, train_loader, val_loader, band_dict)
    models.append(model)
    
    model.eval()
    preds = []
    for X, _ in val_loader:
        with torch.no_grad():
            preds.extend(model(X.to(device))['y_pred'].cpu().numpy().flatten())
    all_preds.append(np.array(preds))

val_targets = []
for _, y in val_loader:
    val_targets.extend(y.numpy())
val_targets = np.array(val_targets)

# Ensemble predictions
mean_preds = np.mean(all_preds, axis=0)
weighted_preds = 0.4*all_preds[0] + 0.35*all_preds[1] + 0.25*all_preds[2]

mean_metrics = calculate_band_mae(mean_preds, val_targets, band_dict)
weighted_metrics = calculate_band_mae(weighted_preds, val_targets, band_dict)

print(f"\nMean ensemble: {mean_metrics['overall_mae']:.4f}")
print(f"Weighted ensemble: {weighted_metrics['overall_mae']:.4f}")

## Summary
| Finding | Result |
|---------|--------|
| Best loss | WeightedL1 |
| DIR losses | No improvement |
| Best VAE | 0.253 |
| Baseline RNN | 0.217 |

**Conclusion:** VAE not suited for imbalanced regression. Simple RNN wins.

## 4. Predictions vs Ground Truth

In [ ]:
# Using utils/visualization.py
from utils.visualization import plot_predictions_scatter

best_preds = all_preds[1]  # weighted_l1 model

# Scatter plot for best model
fig = plot_predictions_scatter(best_preds, val_targets, title='VAE (WeightedL1): Predictions vs Ground Truth')
fig.show()

# Loss function comparison
loss_df = pd.DataFrame(loss_results)
fig_loss = plot_method_comparison(
    [{'method': r['loss'], 'score': r['overall_mae']} for r in loss_results],
    title='Loss Function Comparison: Overall MAE'
)
fig_loss.show()